# Suite TRN — Translating TypeScript to Python

`Transpilers.TypeScriptToPython` translates a TypeScript tree into a Python tree, which never passes through text.
Each case translates programs and checks the Python they print to. The programs of
`conformance/transpilers/typescript-to-python` also run, and print what `<program>.out` holds:
here the programs themselves run in Node, and in Python their translation runs in CPython.

In [ ]:
import { readdirSync, readFileSync } from "node:fs";
import { join } from "node:path";
import { format } from "node:util";
import vm from "node:vm";

import ts from "typescript";

import { transpiled } from "@mbse/programs/Conformance/write";
import { Errors } from "@mbse/programs/Framework";
import { Python314 } from "@mbse/programs/Python";
import { TypeScriptToPython } from "@mbse/programs/Transpilers";
import { TypeScript59 } from "@mbse/programs/TypeScript";

import { assert, equal, raises } from "./support.js";

const PROGRAMS = "../../conformance/transpilers/typescript-to-python";

const translate = (text: string) =>
  Python314.print(TypeScriptToPython.transpile(TypeScript59.parse(text))).replace(/\n+$/, "");

function translates(ts: string, py: string): void {
  equal(translate(ts), py);
}

function fails(ts: string, message: string): void {
  raises(Errors.TranspileError, () => TypeScriptToPython.transpile(TypeScript59.parse(ts)), message);
}

## TRN-01 · Programs keep their meaning: each translation is current, and prints what the TypeScript program prints

In [ ]:
const files = transpiled(PROGRAMS);
for (const program of readdirSync(PROGRAMS).filter((name) => name.endsWith(".ts")).sort()) {
  const stem = program.slice(0, -3);
  assert(files.get(`${stem}.py`) === readFileSync(join(PROGRAMS, `${stem}.py`), "utf8"), `${stem}.py is stale`);
  const js = ts.transpileModule(readFileSync(join(PROGRAMS, program), "utf8"), {
    compilerOptions: { target: ts.ScriptTarget.ES2022, module: ts.ModuleKind.None } }).outputText;
  const lines: string[] = [];
  vm.runInNewContext(js, { console: { log: (...args: unknown[]) => lines.push(`${format(...args)}\n`) } });
  equal(lines.join(""), readFileSync(join(PROGRAMS, `${stem}.out`), "utf8"));
}
console.log(files.size, "programs");

## TRN-02 · Statements: loops, counted ones as `range`, `do ... while`, `switch`, `try`, declarations, assignments

In [ ]:
for (const [ts, py] of [
  ["for (let i = 0; i < n; i++) f(i);", "for i in range(0, n):\n    f(i)"],
  ["for (let i = 1; i <= 10; i += 2) f(i);", "for i in range(1, 11, 2):\n    f(i)"],
  ["for (let i = n; i >= 0; i--) f(i);", "for i in range(n, -1, -1):\n    f(i)"],
  ["for (let i = 0; i < n; i++) { i++; }", "i = 0\nwhile i < n:\n    i += 1\n    i += 1"],
  ["for (let i = 0; i < 3; i *= 2) f(i);", "i = 0\nwhile i < 3:\n    f(i)\n    i *= 2"],
  ["for (i = 0, j = 1; i < 3; i++, j++) f(i);", "i = 0\nj = 1\nwhile i < 3:\n    f(i)\n    i += 1\n    j += 1"],
  ["for (;;) { break; }", "while True:\n    break"],
  ["for (let i = 0; i < n; ) { i += 1; }", "i = 0\nwhile i < n:\n    i += 1"],
  ["while (a) b();", "while a:\n    b()"],
  ["do { a(); } while (b);", "while True:\n    a()\n    if not b:\n        break"],
  ["do a(); while (!b);", "while True:\n    a()\n    if b:\n        break"],
  ["for (const [k, v] of pairs) f(k, v);", "for k, v in pairs:\n    f(k, v)"],
  ["for (const k in o) f(k);", "for k in vars(o):\n    f(k)"],
  ["for (x of xs) f(x);", "for x in xs:\n    f(x)"],
  ["switch (f()) { case 1: a(); break; default: b(); }",
   "_subject1 = f()\nif _subject1 == 1:\n    a()\nelse:\n    b()"],
  ["switch (x) { case 1: return; }", "if x == 1:\n    return"],
  ["try { a(); } catch { b(); }", "try:\n    a()\nexcept Exception:\n    b()"],
  ["try { a(); } finally { b(); }", "try:\n    a()\nfinally:\n    b()"],
  ["{ a(); b(); }", "a()\nb()"],
  [";", ""],
  ["let a, b = 2;", "a = None\nb = 2"],
  ["let [x, , ...rest] = xs;", "x, _, *rest = xs"],
  ["const n: number = 1;", "n: float = 1"],
  ["let s: string;", "s = None"],
  ["x += 1; x -= 1; x *= 2; x **= 2; x %= 3; x <<= 1; x >>= 1; x &= 1; x |= 1; x ^= 1; x /= 2;",
   "x += 1\nx -= 1\nx *= 2\nx **= 2\nx %= 3\nx <<= 1\nx >>= 1\nx &= 1\nx |= 1\nx ^= 1\nx /= 2"],
  ["a = b = c = 0; x++; --y; delete o.k; delete o[k];", "a = b = c = 0\nx += 1\ny -= 1\ndel o.k\ndel o[k]"],
  ["o.a.b = 1; o[i] = 2; [a, b] = [b, a];", "o.a.b = 1\no[i] = 2\na, b = [b, a]"],
  ["/* block\n * comment */\nx; // trailing", "# block\n# comment\nx  # trailing"],
  ["[a, , ...b] = xs;", "a, _, *b = xs"],
  ["(a) = 1;", "a = 1"],
  ["for (let i = 0; j < 3; i++) { f(i); }", "i = 0\nwhile j < 3:\n    f(i)\n    i += 1"],
  ["if (a) { f(); } else { g(); }", "if a:\n    f()\nelse:\n    g()"],
  ["while (a) b();", "while a:\n    b()"],
  ["if (a) f(); else if (b) g(); else h();", "if a:\n    f()\nelif b:\n    g()\nelse:\n    h()"],
  ["x; /* one\n * two */", "x  # one\n# two"],
  ["for (let i = 0; i < n; i++) { i = 2; }", "i = 0\nwhile i < n:\n    i = 2\n    i += 1"],
  ["for (let i = 0; i <= n; i++) { f(i); }", "for i in range(0, n + 1):\n    f(i)"],
  ["while (a) {}", "while a:\n    pass"],
  ["for (let i = n; i >= m; i--) { f(i); }", "for i in range(n, m - 1, -1):\n    f(i)"],
] as [string, string][]) translates(ts, py);
console.log("ok");

## TRN-03 · Functions: parameters, async and generators, lambdas, hoisted functions, `global` and `nonlocal`

In [ ]:
for (const [ts, py] of [
  ["function f(a: number, b = 2, c?: string, ...rest: number[]): number { return a; }",
   "def f(a: float, b=2, c: str = None, *rest) -> float:\n    return a"],
  ["async function f() { await g(); }", "async def f():\n    await g()"],
  ["function* g() { yield; yield 1; yield* h(); }", "def g():\n    yield\n    yield 1\n    yield from h()"],
  ["function f() {}", "def f():\n    pass"],
  ["const f = (a, b) => a + b;", "f = lambda a, b: a + b"],
  ["const g = async () => { await h(); };", "async def g():\n    await h()"],
  ["const h = function named() { return named; };", "def h():\n    return named"],
  ["xs.map(function (x) { return x + 1; });", "def _function1(x):\n    return x + 1\n\n\nlist(map(_function1, xs))"],
  ["xs.map((x) => { return x * 2; });", "def _function1(x):\n    return x * 2\n\n\nlist(map(_function1, xs))"],
  ["function f(this: Window) { }", "def f():\n    pass"],
  ["function outer() { let n = 0; function inner() { n++; total += n; } inner(); return n; }\nlet total = 0;",
   "def outer():\n    n = 0\n\n    def inner():\n        global total\n        nonlocal n\n        n += 1\n" +
   "        total += n\n\n    inner()\n    return n\n\n\ntotal = 0"],
  ["function f() { var x = 1; { x = 2; } return x; }", "def f():\n    x = 1\n    x = 2\n    return x"],
  ["declare function d(): void; function o(a: string): void; function o(a: any) {}",
   "from typing import Any\n\n\ndef o(a: Any):\n    pass"],
  ["const _function1 = 1; g(function () { return _function1; });",
   "_function1 = 1\n\n\ndef _function2():\n    return _function1\n\n\ng(_function2)"],
  ["g(async (x) => x);", "async def _function1(x):\n    return x\n\n\ng(_function1)"],
  ["let n = 0; function f() { n = 1; n = 2; }", "n = 0\n\n\ndef f():\n    global n\n    n = 1\n    n = 2"],
  ["function f() { undeclared = 1; }", "def f():\n    undeclared = 1"],
  ["g(() => { return 1; });", "def _function1():\n    return 1\n\n\ng(_function1)"],
  ["g(function named() { return 1; });", "def named():\n    return 1\n\n\ng(named)"],
] as [string, string][]) translates(ts, py);
console.log("ok");

## TRN-04 · Classes: fields, parameter properties, constructors, accessors, static and abstract members, `super`

In [ ]:
for (const [ts, py] of [
  ["class A { x = 1; y: number; static s = 2; static t: string = \"t\"; constructor(public p: number, q = 1) { this." +
   "y = q; } }",
   "class A:\n    y: float\n    s = 2\n    t: str = \"t\"\n\n    def __init__(self, p: float, q=1):\n" +
   "        self.p = p\n        self.x = 1\n        self.y = q"],
  ["class B extends A { m(): number { return super.m() + 1; } static make(): B { return new B(1); } }",
   "class B(A):\n    def m(self) -> float:\n        return super().m() + 1\n\n    @staticmethod\n" +
   "    def make() -> B:\n        return B(1)"],
  ["class C { get v(): number { return 1; } set v(x: number) {} toString(): string { return \"c\"; } }",
   "class C:\n    @property\n    def v(self) -> float:\n        return 1\n\n    @v.setter\n" +
   "    def v(self, x: float):\n        pass\n\n    def __str__(self) -> str:\n        return \"c\""],
  ["abstract class D { abstract run(n: number): void; #k = 0; [key: string]: unknown; }",
   "class D:\n    def __init__(self):\n        self._k = 0\n\n    def run(self, n: float) -> None:\n" +
   "        raise NotImplementedError"],
  ["class E extends Base {}", "class E(Base):\n    pass"],
  ["class F { m() { const g = () => this.x; return g(); } }",
   "class F:\n    def m(self):\n        g = lambda: self.x\n        return g()"],
  ["class G {}", "class G:\n    pass"],
  ["class H { // only a comment\n}", "class H:  # only a comment\n    pass"],
  ["declare class Z {} declare const z: number; declare enum Q {}", ""],
  ["class A { f(x: number): void; f(x: any) { return x; } }",
   "from typing import Any\n\n\nclass A:\n    def f(self, x: Any):\n        return x"],
  ["class A {} class B extends A { x = 1; }",
   "class A:\n    pass\n\n\nclass B(A):\n    def __init__(self, *args):\n        super().__init__(*args)\n" +
   "        self.x = 1"],
  ["class A { static s; static t: number; }", "class A:\n    s = None\n    t: float = None"],
  ["class A { constructor(private x = 1) {} }", "class A:\n    def __init__(self, x=1):\n        self.x = x"],
  ["class A { constructor() {} }", "class A:\n    def __init__(self):\n        pass"],
] as [string, string][]) translates(ts, py);
console.log("ok");

## TRN-05 · Expressions: operators, `??`, strings, numbers, objects, and the globals and methods mapped by name

In [ ]:
for (const [ts, py] of [
  ["x = a && b || !c; y = a ? b : c; z = -a + +b * ~c; w = a ** b % c; v = a / b;",
   "x = a and b or not c\ny = b if a else c\nz = -a + +b * ~c\nw = a ** b % c\nv = a / b"],
  ["f(a === b, a !== b, a == null, a != undefined, a < b, a >= b, k in o, a instanceof B);",
   "f(a == b, a != b, a is None, a is not None, a < b, a >= b, k in o, isinstance(a, B))"],
  ["x = a ?? b; y = f() ?? g();",
   "x = a if a is not None else b\ny = _value1 if (_value1 := f()) is not None else g()"],
  ["x = [1, ...ys]; f(...args); n = 1_000 + 0x1F + 017 + 9n;", "x = [1, *ys]\nf(*args)\nn = 1_000 + 0x1F + 0o17 + 9"],
  ["s = 'a\\'b' + \"c\\u{1F600}\" + `t${x}{y}\\``;", "s = 'a\\'b' + \"c\\U0001F600\" + f\"t{x}{{y}}`\""],
  ["o = { a: 1, 'b': 2, c, class: 3 };",
   "from types import SimpleNamespace\no = SimpleNamespace(a=1, b=2, c=c, class_=3)"],
  ["x = undefined ?? null; y = NaN; z = Infinity; t = true && false;",
   "import math\nx = None if None is not None else None\ny = math.nan\nz = math.inf\nt = True and False"],
  ["if ((n = next()) > 0) f(n);", "if (n := next()) > 0:\n    f(n)"],
  ["x = <T>y; z = w as unknown as number; q = p!; r = s satisfies T;", "x = y\nz = w\nq = p\nr = s"],
  ["console.log(1); console.error(2); console.warn(3);",
   "import sys\nprint(1)\nprint(2, file=sys.stderr)\nprint(3, file=sys.stderr)"],
  ["x = [Math.floor(a), Math.ceil(a), Math.sqrt(a), Math.trunc(a), Math.abs(a), Math.max(a, b), Math.min(a, b), Math" +
   ".random(), Math.pow(a, 2), Math.log(a), Math.exp(a), Math.PI, Math.E];",
   "import math\nimport random\nx = [math.floor(a), math.ceil(a), math.sqrt(a), math.trunc(a), abs(a), max(a, b), m" +
   "in(a, b), random.random(), a ** 2, math.log(a), math.exp(a), math.pi, math.e]"],
  ["x = [JSON.stringify(o), JSON.parse(s), String(1), Number('2'), Boolean(0), parseInt('3'), parseFloat('4.5'), Arr" +
   "ay.isArray(a), Object.keys(o), Object.values(o), Object.entries(o)];",
   "import json\nx = [json.dumps(o, separators=(',', ':')), json.loads(s), str(1), float('2'), bool(0), int('3'), f" +
   "loat('4.5'), isinstance(a, list), list(vars(o)), list(vars(o).values()), list(vars(o).items())]"],
  ["x = [a.length, s.size, a.includes(1), a.join(), a.join(' '), a.slice(1), a.slice(1, 2), a.slice(), a.toString()," +
   " m.has(k)];",
   "x = [len(a), len(s), 1 in a, ','.join(map(str, a)), ' '.join(map(str, a)), a[1:], a[1:2], a[:], str(a), k in m]"],
  ["x = [s.toUpperCase(), s.toLowerCase(), s.trim(), s.startsWith('a'), s.endsWith('b'), m.entries(), a.reduce(f), a" +
   ".reduce(f, 0), a.map(f), a.filter(f), a.map(x => x + 1), a.filter(x => x > 1)];",
   "import functools\nx = [s.upper(), s.lower(), s.strip(), s.startswith('a'), s.endswith('b'), m.items(), functool" +
   "s.reduce(f, a), functools.reduce(f, a, 0), list(map(f, a)), list(filter(f, a)), [x + 1 for x in a], [x for x in" +
   " a if x > 1]]"],
  ["a.push(1); a.push(1, 2); a.push(...b); m.set(k, v); a.forEach(x => f(x)); a.forEach(x => { f(x); });",
   "a.append(1)\na.extend([1, 2])\na.extend([*b])\nm[k] = v\nfor x in a:\n    f(x)\nfor x in a:\n    f(x)"],
  ["const console = { log: (x) => x }; console.log(1); class Q { length = 1; push(x) {} } q.length; q.push(1);",
   "from types import SimpleNamespace\nconsole = SimpleNamespace(log=lambda x: x)\nconsole.log(1)\n\n\nclass Q:\n" +
   "    def __init__(self):\n        self.length = 1\n\n    def push(self, x):\n        pass\n\n\nq.length\n" +
   "q.push(1)"],
  ["try { f(); } catch (e) { g(e.message); } const e = { message: 'm' }; h(e.message);",
   "from types import SimpleNamespace\ntry:\n    f()\nexcept Exception as e:\n    g(str(e))\n" +
   "e = SimpleNamespace(message='m')\nh(e.message)"],
  ["x = class_ + list + print;", "x = class_ + list_ + print_"],
  ["let print = 1, len = 2;", "print_ = 1\nlen_ = 2"],
  ["xs.reduce(f);", "import functools\nfunctools.reduce(f, xs)"],
  ["o = { \"a\": 1 };", "from types import SimpleNamespace\no = SimpleNamespace(a=1)"],
] as [string, string][]) translates(ts, py);
console.log("ok");

## TRN-06 · Types as annotations, enums, type aliases, imports and exports

In [ ]:
for (const [ts, py] of [
  ["o.class = 1; o.#p; x = o.from;", "o.class_ = 1\no._p\nx = o.from_"],
  ["let a: number = 1, b: string = '', c: boolean = true, d: bigint = 1n, e: object = {}, f: unknown = 1, g: any = 1" +
   ";",
   "from types import SimpleNamespace\nfrom typing import Any\na: float = 1\nb: str = ''\nc: bool = True\n" +
   "d: int = 1\ne: object = SimpleNamespace()\nf: object = 1\ng: Any = 1"],
  ["let a: number[] = [], b: Array<string> = [], c: readonly number[] = [], d: [number, string] = [1, ''], e: Set<nu" +
   "mber> = s;",
   "a: list[float] = []\nb: list[str] = []\nc: list[float] = []\nd: tuple[float, str] = [1, '']\ne: set[float] = s"],
  ["let a: number | null = null, b: 'x' | 'y' = 'x', c: Map<string, number> = m, d: Record<string, number> = r, e: P" +
   "romise<number> = p;",
   "from typing import Literal\na: float | None = None\nb: Literal['x'] | Literal['y'] = 'x'\n" +
   "c: dict[str, float] = m\nd: dict[str, float] = r\ne = p"],
  ["let a: Point = p, b: Box<number> = q, c: (x: number) => number = f, d: void = undefined, e: (number) = 1, f: Rea" +
   "donlyArray<number> = [];",
   "a: Point = p\nb: Box[float] = q\nc = f\nd: None = None\ne: float = 1\nf: list[float] = []"],
  ["type Id = number | string; type Pair<T> = [T, T]; type Fn = () => void; interface I { a: number }",
   "type Id = float | str"],
  ["enum E { A, B = 5, C } enum S { X = 'x', Y = \"y\" } const enum K { One = 1 } enum Empty {}",
   "from enum import IntEnum, StrEnum\n\n\nclass E(IntEnum):\n    A = 0\n    B = 5\n    C = 6\n\n\n" +
   "class S(StrEnum):\n    X = 'x'\n    Y = \"y\"\n\n\nclass K(IntEnum):\n    One = 1\n\n\nclass Empty(IntEnum):\n" +
   "    pass"],
  ["import { a, b as c, type T } from './mod'; import * as ns from '../pkg/sub-mod.js'; import 'side'; import * as t" +
   "op from 'lib'; import type { U } from 'u'; import './rel';",
   "from .mod import a, b as c\nfrom ..pkg import sub_mod as ns\nimport side\nimport lib as top\nfrom . import rel"],
  ["export const x = 1; export function f() {} export { x as y }; export default class K {} export interface I {}",
   "x = 1\n\n\ndef f():\n    pass\n\n\nclass K:\n    pass"],
  ["let x: number | (() => void) = 1;", "x = 1"],
  ["let x: Array<() => void> = [];", "x = []"],
  ["import { a } from \"lib\";", "from lib import a"],
  ["let x: (() => void)[] = [];", "x = []"],
  ["let x: [] = [];", "x = []"],
  ["let x: [() => void] = [];", "x = []"],
] as [string, string][]) translates(ts, py);
console.log("ok");

## TRN-07 · What is not supported fails, at the node's path

In [ ]:
for (const [ts, message] of [
  ["x?.y;", "body[0].expression: ChainExpression is not supported"],
  ["l: for (;;) { break l; }", "body[0]: LabeledStatement is not supported"],
  ["x = typeof y;", "body[0].expression.right: the operator typeof is not supported"],
  ["x = void 0;", "body[0].expression.right: the operator void is not supported"],
  ["f(delete o.k);", "body[0].expression.arguments[0]: the operator delete is not supported"],
  ["new Foo();", "body[0].expression: new Foo is not supported"],
  ["switch (x) { case 1: a(); case 2: b(); }", "body[0].cases[0]: a case that falls through is not supported"],
  ["switch (x) { case 1: if (a) break; b(); break; }",
   "body[0].cases[0].consequent[0].consequent: break within a case is not supported"],
  ["x = /a/g;", "body[0].expression.right: a regular expression is not supported"],
  ["async function* g() {}", "body[0]: an async generator is not supported"],
  ["function f() { return this; }", "body[0].body.body[0].argument: this outside a method is not supported"],
  ["x = this;", "body[0].expression.right: this outside a method is not supported"],
  ["using r = f();", "body[0]: using is not supported"],
  ["do { if (a) continue; } while (b);", "body[0].body.body[0].consequent: continue in this loop is not supported"],
  ["async function f() { for await (const x of xs) {} }", "body[0].body.body[0]: for await is not supported"],
  ["x = { ...o };", "body[0].expression.right.properties[0]: a spread in an object literal is not supported"],
  ["x = { [k]: 1 };", "body[0].expression.right.properties[0]: this property in an object literal is not supported"],
  ["x = { 'a-b': 1 };", "body[0].expression.right.properties[0]: a key that is not a name is not supported"],
  ["x = [1, , 2];", "body[0].expression.right.elements[1]: a hole in an array is not supported"],
  ["f(o.x = 1);", "body[0].expression.arguments[0]: an assignment within an expression is not supported"],
  ["const { a } = o;", "body[0].declarations[0].id: ObjectPattern as a binding is not supported"],
  ["function f({ a }) {}", "body[0].params[0]: ObjectPattern as a parameter is not supported"],
  ["x = #p in o;", "body[0].expression.right: #name in is not supported"],
  ["x = a >>> 1;", "body[0].expression.right: the operator >>> is not supported"],
  ["x >>>= 1;", "body[0].expression: the operator >>>= is not supported"],
  ["import d from 'm';", "body[0].specifiers[0]: a default import is not supported"],
  ["import { 'a b' as c } from 'm';", "body[0].specifiers[0]: importing a name that is a string is not supported"],
  ["export { a } from 'm';", "body[0]: a re-export is not supported"],
  ["export default 1;", "body[0]: a default export of a value is not supported"],
  ["try {} catch ({ message }) {}", "body[0].handler.param: a pattern in catch is not supported"],
  ["enum E { A = 1 + 1 }", "body[0].body.members[0].initializer: an enum member's value must be a literal"],
  ["enum E { A = 'a', B }", "body[0].body.members[1]: a member of a string enum needs a value"],
  ["enum E { ['a'] = 1 }", "body[0].body.members[0]: an enum member must be named by a name"],
  ["@d class C {}", "body[0].decorators[0]: a decorator is not supported"],
  ["import { a } from '@scope/pkg';", "body[0].source: the module '@scope/pkg' has no Python name"],
  ["import { a } from '..';", "body[0].source: the module '..' has no Python name"],
  ["class C { 'a' = 1; }", "body[0].body.body[0]: a member named by a literal is not supported"],
  ["class C { [k] = 1; }", "body[0].body.body[0]: a computed member name is not supported"],
  ["class C { static { } }", "body[0].body.body[0]: StaticBlock is not supported"],
  ["const f = parseInt;", "body[0].declarations[0].init: the global parseInt is not supported here"],
  ["x = Math.floor;", "body[0].expression.right: Math.floor is not supported"],
  ["Math.foo(1);", "body[0].expression: Math.foo() is not supported"],
  ["x = Math.missing;", "body[0].expression.right: Math.missing is not supported"],
  ["namespace N {}", "body[0]: TSModuleDeclaration is not supported"],
  ["x = class {};", "body[0].expression.right: ClassExpression is not supported"],
  ["x = f`t`;", "body[0].expression.right: TaggedTemplateExpression is not supported"],
  ["f(() => function () { return 1; });",
   "body[0].expression.arguments[0].body: a function with a body within a lambda is not supported"],
  ["x = (a, b);", "body[0].expression.right.expression: SequenceExpression is not supported"],
  ["for (const { a } of xs) {}", "body[0].left.declarations[0].id: ObjectPattern as a binding is not supported"],
  ["x = a!.b?.c;", "body[0].expression.right: ChainExpression is not supported"],
  ["const f = () => x = 1;", "body[0].declarations[0].init.body: an assignment within an expression is not supported"],
  ["xs.forEach(function (x) {});",
   "body[0].expression.arguments[0]: forEach takes an arrow function of one parameter here"],
  ["x = [new Point(1, 2)];", "body[0].expression.right.elements[0]: new Point is not supported"],
  ["xs.length = 0;", "body[0].expression.left: MemberExpression as a target is not supported"],
  ["f(([a]) => a);", "body[0].expression.arguments[0].params[0]: a lambda's parameters must be names"],
  ["function f() { continue; }", "body[0].body.body[0]: ContinueStatement outside of a loop is not supported"],
  ["for (const x of xs) { f(function () { break; }); }",
   "body[0].body.body[0].expression.arguments[0].body.body[0]: BreakStatement outside of a loop is not supported"],
  ["xs.map(f, 1);", "body[0].expression: this call's arguments are not supported"],
  ["continue;", "body[0]: ContinueStatement outside of a loop is not supported"],
  ["o = { 1: 2 };", "body[0].expression.right.properties[0]: a key that is not a name is not supported"],
] as [string, string][]) fails(ts, message);
console.log("ok");
equal(new Errors.TranspileError("message", "").message, "message");